In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("preatcher/standard-ocr-dataset")

print("Path to dataset files:", path)

# **Here lets see whats inside the data that we have**

Here we have traing and test data folders

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

In [ ]:
trainig_data_path = '/kaggle/input/standard-ocr-dataset/data/training_data'
test_data_path = '/kaggle/input/standard-ocr-dataset/data/testing_data'

train_folder = os.listdir(trainig_data_path)
test_folder = os.listdir(test_data_path)

data = []

# lets see the data inside the folders
for lable in train_folder:
    lable_path = os.path.join(trainig_data_path ,  lable)

    # loop throgh the folders and get the images
    # check the folder is empty or not
    if os.path.isdir(lable_path):
        # loop
        for image_name in os.listdir(lable_path):
            img_path =  os.path.join(lable_path , image_name)
            # store both image and lable path
            data.append((img_path ,lable))


# convert the data into a dataframe
df = pd.DataFrame(data ,columns=['image_path', 'label'])

In [ ]:
df.head()

Now lets see the images how the look like Here i need to see 10 images from each lable

In [ ]:
df.describe()

In [ ]:
for label in df['label'].unique():
    label_images = df[df['label'] == label].sample(30 , random_state = 42)
    plt.figure(figsize = (20 ,5))
    plt.suptitle(f'label: {label}' , fontsize = 20)


    for i , row in enumerate(label_images.itertuples() , 1):
        img =  mpimg.imread(row.image_path)
        plt.subplot(1 , 30 ,i)
        plt.imshow(img, cmap='gray')  # use cmap='gray' for grayscale images
        plt.axis('off')

    plt.show()
        

In [ ]:
df.info()

In [ ]:
!pip install tensorflow pandas numpy matplotlib

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import numpy as np

# Convert all the charactors in to integers

Because later we need to refer to the classes as we cannot use the letter we need numbers

In [ ]:
import string

charactors = list(string.digits + string.ascii_uppercase)
print(charactors)

# as we need to convert all these charactors in to numbers
char_to_int = {char : i for i , char in enumerate(charactors)}
print(char_to_int)


# number of unique classes
num_classes = len(char_to_int)

**Data loader as bathced of 32**

In [ ]:
# image size
IMG_SIZE = ( 32, 32)

def load_images(df):
    images = []
    lables= []

    for _ ,row in df.iterrows():
        image_path =  row['image_path']
        label =  row['label']

        # conbvert all the labels in to integers using the mapping we created
        lable_int =  char_to_int[label]

        # load the images and convert to array
        img = load_img(image_path , target_size = IMG_SIZE ,color_mode = "grayscale")

        img_array = img_to_array(img)

        # normalize the array
        img_array =  img_array/255

        images.append(img_array)

        lables.append(lable_int)

    # print(images)
    # print(lables)
    return np.array(images) , np.array(lables)


# Create X and y train from the images and lables usinf the data loader

In [ ]:
X_train , y_train =  load_images(df)

print(X_train.shape) # (20628, 32, 32, 1)

# Here we need to reshape the images in to the CNN input format [num_samples, height, width, channels ]
X_train =  X_train.reshape(-1 , 32 , 32 , 1)

print(X_train.shape)

# one-hot encoding of the lables

Neural networks with ***categorical_crossentropy*** loss expect labels in a one-hot encoded format.
Neural networks output probabilities for each class (via softmax). 

Example output

[0.1, 0.7, 0.1, 0.1] The target label must match this format (one-hot).


In [ ]:
# Convert labels to categorical (one-hot encoding)
y_train = to_categorical(y_train, num_classes)

print(y_train)

In [ ]:
print("Unique training labels:", np.unique(y_train, return_counts=True))

In [ ]:
from sklearn.model_selection import train_test_split

# reverse onehot encoding
y_int_lables = np.argmax(y_train , axis=1)  #axis=1 → find the index of the maximum value along each row

# Split data while maintaining class distribution
X_train ,X_val , y_train_int , y_val_int = train_test_split(X_train , y_int_lables , test_size = 0.2 , stratify=y_int_lables , random_state = 42)

# convert back to onehot encoding
y_train = to_categorical(y_train_int , num_classes)
y_val= to_categorical(y_val_int , num_classes)


# Verify class distribution
print("Training labels distribution:", np.unique(y_train_int, return_counts=True))
print("Validation labels distribution:", np.unique(y_val_int, return_counts=True))

# Create the CNN model architecture

In [ ]:
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.regularizers import l2


# model
model = Sequential([

    # conv layer
    Conv2D(
        32,                          # filters
        (3, 3),                      # kernel_size
        activation='relu',           # activation function -applied to the output of the convolution.
        kernel_regularizer=l2(0.001),# regularization - applied to the output of the convolution.0.001 is the regularization strength (lambda).
        input_shape=(32, 32, 1)      # input shape
    ),

    BatchNormalization(),  # BatchNormalization() keeps the network stable and helps it train faster by normalizing activations, then letting the model re-scale and re-shift them with learnable parameters.
    MaxPooling2D((2, 2)),
    Dropout(0.3),

    # conv layer
    Conv2D(64, (3, 3), activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.3),

    Flatten(),
    Dense(128, activation='relu', kernel_regularizer=l2(0.001)), # fully connected
    Dropout(0.5),

    Dense(num_classes, activation='softmax')  # Output layer

])

model.summary()

### Creating the optimizer and loss function

In [ ]:
from tensorflow.keras.optimizers import Adam

model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

Data augmentation

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1
)


# Fit generator to training data
datagen.fit(X_train)

### Train

In [ ]:
history = model.fit(datagen.flow(X_train, y_train  , batch_size = 32) , epochs = 20 , validation_data = (X_val , y_val))

### Evaluate the model

In [ ]:
plt.plot(history.history['accuracy'] , label = 'Train accuracy')
plt.plot(history.history['val_accuracy'] , label = 'validation accuracy')

plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.legend()
plt.show()

In [ ]:
# save the model
model.save("/kaggle/working/ocr_character_model.h5")

### Now lets load the testing images and try the model

In [ ]:
from tensorflow.keras.preprocessing.image import load_img , img_to_array

IMG_SIZE = (32 ,32)

testing_data_path = '/kaggle/input/standard-ocr-dataset/data/testing_data'

# load the data from the testing folder

def load_test_imgs(testing_img_path):
    test_images = []
    test_image_paths = []
    
    for lables in os.listdir(testing_data_path):
        # get all the images in the lables folders
        # path
        image_folder_path = os.path.join(testing_img_path , lables)

        if (os.path.isdir(image_folder_path)):

            for image in os.listdir(image_folder_path):
                image_path = os.path.join(image_folder_path , image)

                # Load image and preprocess
                image = load_img(image_path , color_mode='grayscale',target_size=IMG_SIZE)

                # normalize the image
                image_array = img_to_array(image) /255

                # Store filename for reference
                test_images.append(image_array)
                test_image_paths.append(image_path)

    return np.array(test_images) , test_image_paths
                



In [ ]:
X_test , file_names = load_test_imgs(testing_data_path)  

In [ ]:
# reshape the images that mathch the CNN
x_test = X_test.reshape(-1 , 32 , 32, 1)

In [ ]:
# predictions
y_pred =  model.predict(X_test)



In [ ]:
print(np.argmax(y_pred , axis=1))

In [ ]:
# we need to get the exact class from one hot encoding --> as we are specifying the axis =1 we are tellign that we need
# to get the column level max this will return us the class with the max probability
y_pred_labels =  np.argmax(y_pred , axis = 1)

# cconvert the integers back to chars (we need to reverse)

int_to_char = {i : char for char, i in char_to_int.items()}

# we pass all the llables to the reverse map and turn that in to a list
predicted_chars = [int_to_char[i] for i in y_pred_labels]

# Print some predictions
for i in range(100):  # Show 10 sample predictions
    print(f"Image: {file_names[i]} → Predicted Character: {predicted_chars[i]}")

### We can now eveluate the model using the loss by getting the predicted and actual character

In [ ]:
#  explanation >> for a path name -> /kaggle/input/standard-ocr-dataset/data/testing_data/2/29284.png 
# here the os.path.dirname --->/kaggle/input/standard-ocr-dataset/data/testing_data/2/
# here we need the 2 only from the path
# so we can get it using the basename ---> 2

actual_lables = [os.path.basename(os.path.dirname(f)) for f in file_names]

# convert to np array
actual_lables = np.array(actual_lables , dtype = str)
predicted_lables = np.array(predicted_chars , dtype =str)

# upper case both for more consistancy
actual_lables = np.char.upper(actual_lables)
predicted_lables = np.char.upper(predicted_lables)

# compair

correct = np.sum (predicted_lables == actual_lables)

total = len(actual_lables)

accuracy =  (correct/total) *100

print(f'test accuracy: {accuracy}' )

### Lets visualize

In [ ]:
import random

random_indices =  random.sample(range(len(X_test)) , 100)

fig , axes =  plt.subplots(10, 10, figsize=(20,20))

for i , ax in zip(rando_indices , axes.flatten()):
    ax.imshow(X_test[i].reshape(32,32) , cmap = 'gray')
    ax.set_title(f'predicted : {predicted_chars[i]}')
    ax.axis("off")

plt.tight_layout()
plt.show()

### On Our images

In [ ]:
# custum image
custom_image_path ="/kaggle/input/test-data/simple_a.png" 
IMG_SIZE = (32,32)

# load the model
model = tf.keras.models.load_model("/kaggle/working/ocr_character_model.h5")

# preprocess the images
image = load_img(custom_image_path ,target_size = IMG_SIZE , color_mode = "grayscale")
image_array = img_to_array(image) / 255

image_array = image_array.reshape(1 , 32,32,1)

# predict

predicted_lable = model.predict(image_array)

# onehot encode and convert to char
predicted_lable =np.argmax(predicted_lable , axis =1)

print(predicted_lable)

predicted_char = [int_to_char[i] for i in predicted_lable ]

# display
plt.imshow(image ,cmap ="gray")
plt.title(f"Predicted Character: {predicted_char}")
plt.axis("off")
plt.show()

print(f"✅ Model Prediction: {predicted_char}")
